# Part 2 — Fine-tune the best model with NeSy + labs

Loads the **best Part-1 checkpoint from Drive** and makes it obey the temporal-logic traffic rules: **predicates** (Step A) → **FSM safety shield** (Step B, zero retraining) → **logic-shaped reward fine-tune** (Step C). Evaluates four configs with a **per-rule violation rate** from an independent monitor, and saves the **NeSy checkpoint + `part2_nesy.mp4`** to Drive. The predicate/shield/reward modules built here are reused by Part 3.

## 1. Setup + load `part1_best` from Drive

In [ ]:
import os, warnings, logging
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', message=r'.*Gym has been unmaintained.*')
logging.getLogger('gym').setLevel(logging.ERROR)

from google.colab import drive
drive.mount('/content/drive')

TOKEN = None
try:
    from google.colab import userdata
    TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    TOKEN = None
if not TOKEN:
    import getpass
    TOKEN = getpass.getpass('GitHub token (input hidden): ')
assert TOKEN, 'No token provided.'
os.environ['GITHUB_TOKEN'] = TOKEN
print('Token loaded (not displayed).')

In [ ]:
%%bash
set -e
GH_USER=silvergjeka22
REPO=nesy-highway-driving
curl -fsSL -H "Authorization: token ${GITHUB_TOKEN}" \
  "https://raw.githubusercontent.com/${GH_USER}/${REPO}/main/bash/setup_colab.sh" \
  -o /content/setup_colab.sh
bash /content/setup_colab.sh

In [ ]:
import sys, json
sys.path.insert(0, '/content/nesy-highway-driving')
%cd /content/nesy-highway-driving
from utils import load_config, drive_path, save_json
cfg = load_config('configs/highway.yaml')
from pyvirtualdisplay import Display
_display = Display(visible=0, size=(900, 480)); _display.start()

from agents.baselines import load_model
TAG = json.load(open(drive_path(cfg, 'metrics', 'part1_best.json')))['tag']
BEST = drive_path(cfg, 'checkpoints', f'part1_best_{TAG}.zip')
baseline = load_model(BEST, TAG)
print('loaded best Part-1 model:', TAG)

## 2. Step A — Predicates (perception → logic)
Ground the observation into truth-valued predicates the rules are written over (safe-gap via **velocity obstacles, Lab 4**; proximity/off-road via **LIDAR helpers, Lab 2**). Sanity-check on a live scene first.

In [ ]:
from envs.highway_factory import make_env, read_scene
from nesy.roadmap import predicates
env = make_env(cfg)
obs, _ = env.reset(seed=cfg['eval_seeds'][0])
for _ in range(5):
    a, _ = baseline.predict(obs, deterministic=True)
    obs, *_ = env.step(a)
print(predicates(read_scene(env), cfg))
env.close()

## 3. Step B — FSM safety shield (Lab 3), zero retraining
Wrap the frozen best model in the FSM (`CRUISE/FOLLOW/OVERTAKE-LEFT/MERGE/EMERGENCY-STOP`); a manoeuvre violating a **hard constraint** (RG1 safe gap, RG3 speed limit, RI1 no-stop, stay-on-road) is replaced with the safest legal fallback. Clean on/off ablation, with per-rule violations.

In [ ]:
from eval.evaluate import evaluate, record_video
m_baseline = evaluate(baseline, cfg, count_violations=True)
m_shield   = evaluate(baseline, cfg, apply_shield=True, count_violations=True)
save_json(m_baseline, drive_path(cfg, 'metrics', 'p2_baseline.json'))
save_json(m_shield,   drive_path(cfg, 'metrics', 'p2_shield.json'))
print('shield violation rates:', m_shield['summary']['rule_violation_rate'])

## 4. Step C — Logic-shaped reward fine-tune (the actual fine-tune)
Warm-start from the best checkpoint and continue training on `reward - Σ λ_i·violation_i` (soft heuristics RI2/RG4/RG2; lower LR, fewer steps; weights from the YAML). Then evaluate the fine-tuned policy with the shield on.

In [ ]:
from agents.baselines import finetune_logic_reward
nesy_model = finetune_logic_reward(load_model(BEST, TAG), cfg)   # -> checkpoints/part2_nesy.zip
m_reward        = evaluate(nesy_model, cfg, apply_shield=False, count_violations=True)
m_shield_reward = evaluate(nesy_model, cfg, apply_shield=True,  count_violations=True)
save_json(m_reward,        drive_path(cfg, 'metrics', 'p2_reward.json'))
save_json(m_shield_reward, drive_path(cfg, 'metrics', 'p2_shield_reward.json'))

## 5. Four-config comparison (with per-rule violations)

In [ ]:
import pandas as pd
def row(name, m):
    s = m['summary']; v = s.get('rule_violation_rate', {})
    out = {'config': name, 'crash': round(s['crash_rate'], 3),
           'on_road_%': round(s['on_road_pct']['mean'], 1),
           'overtakes': round(s['overtakes']['mean'], 2),
           'return': round(s['return']['mean'], 2)}
    for k in ('RG1', 'RG3', 'RI1', 'RI2', 'RG4'):
        out[k] = round(v.get(k, float('nan')), 3)
    return out
pd.DataFrame([row('Baseline (P1)', m_baseline), row('+ FSM shield', m_shield),
              row('+ Logic reward', m_reward), row('+ Shield+reward', m_shield_reward)])

## 6. Save the NeSy video to Drive

In [ ]:
video = record_video(nesy_model, cfg, drive_path(cfg, 'videos', 'part2_nesy.mp4'), apply_shield=True)
print('Saved: checkpoints/part2_nesy.zip  +  %s' % video)

**Exit criterion (headline).** Equal-or-better overtaking with **provably fewer rule violations** than the baseline (compare the `RG*/RI*` columns), the NeSy checkpoint + **`part2_nesy.mp4`** on Drive. Next: `colab_3_metadrive.ipynb` (Part 3) or `colab_4_race.ipynb` (Part 4).